# NB43: Missing-Flag Korelasyon Analizi — Tüm Paneller

## 0. Ne Yapıyoruz ve Neden

### Problem
Şu anki missing stratejimiz **M3** (>%50 NaN sütunlar için `is_missing_*` flag + tüm sayısala medyan impute) bir
"sezgi + panel-transfer ablasyonu" (NB14) sonucu seçilmişti. Ama flag'leri **hangi sütunlar için üretmemiz
gerektiğini** ve **her flag'in etikete ne kadar bilgi taşıdığını** sistematik ölçmedik. CLAUDE.md'de tek bir
global sayı var: "MASTER'da Label=1'de eksiklik %59.9, Label=0'da %41.2". Bu, panel geneli bir ortalama — hangi
sütunların bu farkı yarattığını, farkın hangi **eksiklik-yoğunluğu bandında** (çok-eksik vs az-eksik) toplandığını
göstermiyor.

### Bu notebook ne yapar (tek cümle)
Her panelde, **sabit ve özdeş sütunlar drop edildikten sonra** kalan her feature için (eksiklik>0 olanlar) bir
missing-flag üretir, o flag'in Label ile korelasyonunu (φ) ve sınıf-bazlı eksiklik oranlarını hesaplar, flag'leri
**>%50 null / <%50 null** olarak iki gruba ayırır ve her grubun **ortalama korelasyonu + korelasyon
dalgalanmasını (fluctuation)** çıkarır. Çıktı: 4 panel × 2 grup = **8 tablo**, hepsi tek bir PDF raporunda.

### Neden önce sabit + özdeş sütun temizliği? (bu revizyonda eklendi)
Diğer modelleme notebook'larında (NB15, NB36–39) CLAUDE.md Sözleşmeler madde 3–4 gereği **sabit sütunlar**
(`nunique<=1`, bilgi taşımaz) ve **özdeş sütun çiftleri** (583 çift, çoğu AL bloklarında — CAT_3==CAT_5 sadece
görünen ucu) drop ediliyor. İlk NB43 çalıştırmasında bu temizlik atlanmıştı: sonuçta "en yüksek φ" listeleri
özdeş AL bloklarının **aynı sinyalin N kez tekrarı** olarak görünmesine yol açtı (örn. MASTER'da AL_1..AL_6 hepsi
birebir aynı miss_ratio ve φ değerine sahipti — 6 bağımsız kanıt değil, 1 sinyalin 6 kopyası). Bu revizyon aynı
temizliği (`get_constant_cols` + `get_duplicate_col_pairs`) uygulayarak her kalan sütunun **kendi başına** bir
sinyal taşıdığından emin oluyor; grup ortalamaları artık şişirilmiş tekrarlardan etkilenmiyor.

### Neden "grup ortalaması", tek tek sütun değil?
Tek bir flag'in φ'si yanıltıcı olabilir (bir sütun güçlü, komşuları zayıf). Asıl karar verici olan **bir sütun
bloğunun birlikte davranışıdır**: "şu ~N sütunu birlikte ortalama alınca ortaya çıkan sinyal budur". Bu yüzden her
grup için:
- **Ortalama |φ|** (grup toplu sinyali) — grup olarak eksiklik ne kadar etikete işaret ediyor?
- **Fluctuation = φ'lerin std'si** (grup-içi tutarsızlık) — sinyal birkaç sütunda mı yoğunlaşmış (yüksek std),
  yoksa homojen mi (düşük std)?

Yüksek ortalama + düşük std = tüm grup tutarlı biçimde etiketle korele (güçlü, güvenilir eksiklik sinyali).
Yüksek ortalama + yüksek std = sinyal birkaç sütunda toplanmış (o sütunları izole et). Düşük ortalama = grup
olarak eksiklik gürültü (flag üretmeye değmez).

### Neden bu iki grup (>%50 / <%50 null)?
M3 eşiği zaten %50. Bu analiz M3'ün eşik seçimini **doğrulayacak ya da çürütecek**:
- **>%50 null grubu** güçlü korele çıkarsa → M3 doğru yerde flag üretiyor.
- **<%50 null grubu da güçlü korele çıkarsa** → M3 bu flag'leri KAÇIRIYOR (eşiği düşürmeliyiz veya seçici flag
  eklemeliyiz). Bu, notebook'un en değerli olası bulgusu.

### Karar kuralı (bu notebook'un çıktısı)

| Bulgu | Anlam | Aksiyon |
|---|---|---|
| <%50 null grubunda ortalama\|φ\| yüksek (>~0.15) | M3 değerli flag'leri kaçırıyor | Flag eşiğini düşür veya φ-tabanlı seçici flag seti kur |
| >%50 null grubunda ortalama\|φ\| yüksek + düşük fluctuation | Eksiklik güçlü, tutarlı sinyal | M3 doğru; bu blok flag'lerini modelde tut |
| Bir grupta yüksek fluctuation | Sinyal birkaç sütunda | O yüksek-φ sütunları ayrı ele al; kalanı drop |
| `AL_16..25` (KANSER) yüksek φ | Bilinen leakage riski teyidi | `AL_MISSINGNESS_LEAKAGE_RISK` ile/olmadan iki model (CLAUDE.md kuralı) |

### Yöntem özet
- **Ön temizlik**: `get_constant_cols()` + `get_duplicate_col_pairs()` panelin tamamı üzerinde (split yok, model
  eğitimi yok → leakage riski yok). Özdeş çiftlerde sütun sırasına göre ilk geçen tutulur.
- **Missing-flag üretimi**: `is_missing_<col> = df[col].isna().astype(int)`, temiz+ham veriden (impute ÖNCESİ).
  Tamamen dolu (%0 eksik) sütunlar analiz dışı.
- **Korelasyon metriği**: φ (phi) katsayısı = ikili-ikili Pearson korelasyonu (`is_missing_<col>` vs `Label`).
  φ>0 → sütun pathogenic'te daha çok eksik; φ<0 → benign'de daha çok eksik.
- **Sınıf-bazlı eksiklik oranları**: `miss_rate_patho`, `miss_rate_benign`, `delta = patho − benign` (φ ile aynı
  yönde, büyüklüğü somutlaştırır).
- **Fluctuation**: `np.std(phi_values_in_group, ddof=1)`.
- **Gruplama**: `miss_ratio > 0.50` → G_HIGH, `0 < miss_ratio <= 0.50` → G_LOW.

### Kritik uyarılar
- Bu bir **ölçüm, tahmin değil** — train/test split YOK, impute YOK. Leakage değil (model eğitmiyoruz).
- **CFTR düşük-güven**: n_benign=21; sınıf-bazlı oranlar çok gürültülü (bkz. CLAUDE.md CFTR protokolü).
- **Degenerate φ guard**: bir sütun bir sınıfta hep eksik/hep dolu ise flag varyansı 0 → φ=0, `note='degenerate'`.
- **AL_16..25 sağlama noktası**: KANSER'de bu blok en yüksek φ'yi vermeli (bilinen leakage riski, CLAUDE.md).


In [1]:
# Cell 1: Imports ve Konfigürasyon
import sys, os, warnings
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from fpdf import FPDF

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if not os.path.exists(os.path.join(PROJECT_ROOT, 'config.py')):
    PROJECT_ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, PROJECT_ROOT as CFG_ROOT
from src.columns_real import (
    ALL_FEATURE_COLS, TARGET_COL, ID_COL, NON_FEATURE_COLS, PANEL_INFO,
    get_missing_mask_col_name, get_high_missing_cols,
    get_constant_cols, get_duplicate_col_pairs,
    AL_MISSINGNESS_LEAKAGE_RISK, AL_HIGH_MISSING_COLS,
)

PROJECT_ROOT = CFG_ROOT
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v26_missing_flag_correlation')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'real_data')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

PANELS = ['MASTER', 'KANSER', 'PAH', 'CFTR']
LOW_CONFIDENCE_PANELS = {'CFTR'}  # n_benign=21 -- CLAUDE.md CFTR protokolü

print(f'SEED={SEED}')
print(f'RESULTS_DIR={RESULTS_DIR}')
print(f'Toplam feature sayisi: {len(ALL_FEATURE_COLS)}')
print(f'Paneller: {PANELS}')


SEED=42
RESULTS_DIR=/Users/tefe/teknofest_model/teknofest_model/results/v26_missing_flag_correlation
Toplam feature sayisi: 351
Paneller: ['MASTER', 'KANSER', 'PAH', 'CFTR']


In [2]:
# Cell 2: Ham Panel Yükleme + Sabit/Özdeş Sütun Temizliği (impute YOK, split YOK)
#
# NB15/NB36-39'da uygulanan temizlik burada da uygulanıyor: sabit sütunlar (nunique<=1,
# bilgi taşımaz) ve özdeş sütun çiftlerinden biri (CAT_3==CAT_5 + AL bloklarındaki 583 çift,
# CLAUDE.md Sözleşmeler madde 3-4) drop edilir. Bu bir ölçüm olduğu için (model eğitmiyoruz,
# split yok) temizlik doğrudan panelin tamamı üzerinde yapılabilir -- leakage riski yok.
#
# Özdeş çiftlerden hangisi tutulur: DataFrame sütun sırasına göre İLK GEÇEN tutulur, geri
# kalanlar drop edilir (deterministik, notebook'lar arası tutarlı).

def get_cols_to_drop(X):
    """Bir panel icin sabit + ozdes sutunlarin birlesik drop listesini dondurur."""
    constant_cols = get_constant_cols(X)
    dup_pairs = get_duplicate_col_pairs(X)

    # ozdes ciftlerde sutun sirasina gore ilkini tut, sonrakini drop et
    dup_drop = set()
    for c1, c2 in dup_pairs:
        idx1, idx2 = X.columns.get_loc(c1), X.columns.get_loc(c2)
        dup_drop.add(c2 if idx1 < idx2 else c1)

    drop_cols = sorted(set(constant_cols) | dup_drop)
    return drop_cols, constant_cols, dup_pairs


def load_panel_raw(name):
    """Panel CSV'sini ham haliyle yukler, sabit+ozdes sutunlari drop eder.
    Flag'ler HAM eksiklikten (drop sonrasi ama impute ONCESI) uretilecek."""
    file_name = PANEL_INFO[name]['file']
    path = os.path.join(DATA_DIR, file_name)
    df = pd.read_csv(path)
    y = df[TARGET_COL].copy()
    feature_cols = [c for c in ALL_FEATURE_COLS if c in df.columns]
    X = df[feature_cols].copy()

    drop_cols, constant_cols, dup_pairs = get_cols_to_drop(X)
    X_clean = X.drop(columns=drop_cols)

    return X_clean, y, {
        'n_raw_cols': X.shape[1],
        'n_constant': len(constant_cols),
        'n_dup_pairs': len(dup_pairs),
        'n_dropped': len(drop_cols),
        'n_clean_cols': X_clean.shape[1],
        'constant_cols': constant_cols,
        'dup_pairs': dup_pairs,
        'dropped_cols': drop_cols,
    }


panel_clean_info = {}
for p in PANELS:
    X_tmp, y_tmp, info_tmp = load_panel_raw(p)
    panel_clean_info[p] = info_tmp
    print(f"{p}: ham={info_tmp['n_raw_cols']} -> sabit={info_tmp['n_constant']}, "
          f"ozdes_cift={info_tmp['n_dup_pairs']} -> drop={info_tmp['n_dropped']} -> "
          f"temiz={info_tmp['n_clean_cols']} | pos={int(y_tmp.sum())}, neg={int((y_tmp == 0).sum())}")


MASTER: ham=351 -> sabit=57, ozdes_cift=583 -> drop=63 -> temiz=288 | pos=2149, neg=782


KANSER: ham=351 -> sabit=69, ozdes_cift=694 -> drop=71 -> temiz=280 | pos=268, neg=120


PAH: ham=351 -> sabit=91, ozdes_cift=886 -> drop=93 -> temiz=258 | pos=310, neg=62


CFTR: ham=351 -> sabit=70, ozdes_cift=696 -> drop=72 -> temiz=279 | pos=90, neg=21


In [3]:
# Cell 3: Flag Istatistikleri Hesaplama
def compute_flag_stats(X, y):
    """Eksiklik>0 olan her sutun icin missing-flag / Label korelasyon istatistikleri."""
    rows = []
    y_arr = y.to_numpy()

    for col in X.columns:
        miss_ratio = X[col].isna().mean()
        if miss_ratio <= 0:
            continue  # tamamen dolu sutun -- flag hep 0, analiz disi

        flag = X[col].isna().astype(int).to_numpy()

        miss_rate_patho = X.loc[y == 1, col].isna().mean()
        miss_rate_benign = X.loc[y == 0, col].isna().mean()
        delta = miss_rate_patho - miss_rate_benign

        # degenerate guard: flag veya label varyansi 0 ise phi tanimsiz
        if flag.std() == 0 or y_arr.std() == 0:
            phi, p_value, note = 0.0, np.nan, 'degenerate'
        else:
            try:
                phi, p_value = pearsonr(flag, y_arr)
                if np.isnan(phi):
                    phi, note = 0.0, 'degenerate'
                else:
                    note = 'ok'
            except Exception:
                phi, p_value, note = 0.0, np.nan, 'degenerate'

        rows.append({
            'col': col,
            'miss_ratio': miss_ratio,
            'miss_rate_patho': miss_rate_patho,
            'miss_rate_benign': miss_rate_benign,
            'delta': delta,
            'phi': phi,
            'abs_phi': abs(phi),
            'p_value': p_value,
            'note': note,
        })

    return pd.DataFrame(rows)


# Hizli dogrulama
_X, _y, _info = load_panel_raw('KANSER')
_stats = compute_flag_stats(_X, _y)
print(f'KANSER: {len(_stats)} sutun icin flag uretildi (eksiklik>0, temiz={_info["n_clean_cols"]} sutun uzerinden)')
print(_stats.sort_values('abs_phi', ascending=False).head(5)[['col', 'miss_ratio', 'phi', 'delta']])


KANSER: 280 sutun icin flag uretildi (eksiklik>0, temiz=280 sutun uzerinden)
      col  miss_ratio       phi     delta
24  AL_25    0.685567  0.567702  0.570274
15  AL_16    0.685567  0.567702  0.570274
23  AL_24    0.685567  0.567702  0.570274
22  AL_23    0.685567  0.567702  0.570274
21  AL_22    0.685567  0.567702  0.570274


In [4]:
# Cell 4: G_HIGH / G_LOW Gruplama
def split_groups(flag_df):
    """miss_ratio > 0.50 -> G_HIGH, 0 < miss_ratio <= 0.50 -> G_LOW."""
    g_high = flag_df[flag_df['miss_ratio'] > 0.50].copy()
    g_low = flag_df[flag_df['miss_ratio'] <= 0.50].copy()
    return g_high, g_low


_g_high, _g_low = split_groups(_stats)
print(f'KANSER -- G_HIGH: {len(_g_high)} sutun, G_LOW: {len(_g_low)} sutun')


KANSER -- G_HIGH: 191 sutun, G_LOW: 89 sutun


In [5]:
# Cell 5: Grup Özet İstatistikleri
def group_summary(group_df):
    """Bir grubun (G_HIGH veya G_LOW) ozet istatistiklerini dondurur."""
    n_cols = len(group_df)
    if n_cols == 0:
        return {
            'n_cols': 0, 'mean_phi': np.nan, 'mean_abs_phi': np.nan,
            'fluctuation': np.nan, 'min_phi': np.nan, 'max_phi': np.nan,
            'mean_delta': np.nan, 'top3_cols': [],
        }

    phi_values = group_df['phi'].to_numpy()
    fluctuation = np.std(phi_values, ddof=1) if n_cols >= 2 else np.nan

    top3 = group_df.sort_values('abs_phi', ascending=False).head(3)['col'].tolist()

    return {
        'n_cols': n_cols,
        'mean_phi': phi_values.mean(),
        'mean_abs_phi': group_df['abs_phi'].mean(),
        'fluctuation': fluctuation,
        'min_phi': phi_values.min(),
        'max_phi': phi_values.max(),
        'mean_delta': group_df['delta'].mean(),
        'top3_cols': top3,
    }


_high_summary = group_summary(_g_high)
_low_summary = group_summary(_g_low)
print('KANSER G_HIGH ozet:', _high_summary)
print('KANSER G_LOW ozet:', _low_summary)


KANSER G_HIGH ozet: {'n_cols': 191, 'mean_phi': np.float64(0.4163992364335517), 'mean_abs_phi': np.float64(0.4163992364335517), 'fluctuation': np.float64(0.06078270913028417), 'min_phi': np.float64(0.3396051506991966), 'max_phi': np.float64(0.567702200529808), 'mean_delta': np.float64(0.4099202938188637), 'top3_cols': ['AL_20', 'AL_22', 'AL_25']}
KANSER G_LOW ozet: {'n_cols': 89, 'mean_phi': np.float64(0.1943038653754965), 'mean_abs_phi': np.float64(0.25308086642336824), 'fluctuation': np.float64(0.22341648862580962), 'min_phi': np.float64(-0.24583650882051858), 'max_phi': np.float64(0.47684037908792776), 'mean_delta': np.float64(0.225231986136732), 'top3_cols': ['AL_7', 'AL_12', 'AL_8']}


In [6]:
# Cell 6: Panel Calistirma Fonksiyonu
def run_panel(name):
    """Bir panel icin 2-5 arasi adimlari calistirir, sonuclari dict olarak dondurur."""
    X, y, clean_info = load_panel_raw(name)
    flag_df = compute_flag_stats(X, y)
    g_high, g_low = split_groups(flag_df)
    high_summary = group_summary(g_high)
    low_summary = group_summary(g_low)

    low_confidence = name in LOW_CONFIDENCE_PANELS

    return {
        'panel': name,
        'n_rows': len(y),
        'n_pos': int(y.sum()),
        'n_neg': int((y == 0).sum()),
        'clean_info': clean_info,
        'flag_df': flag_df,
        'g_high': g_high,
        'g_low': g_low,
        'high_summary': high_summary,
        'low_summary': low_summary,
        'low_confidence': low_confidence,
    }


panel_results = {}
for p in PANELS:
    panel_results[p] = run_panel(p)
    r = panel_results[p]
    lc_tag = ' [DUSUK-GUVEN: n_benign=21]' if r['low_confidence'] else ''
    print(f"{p}{lc_tag}: temiz_sutun={r['clean_info']['n_clean_cols']} "
          f"(drop={r['clean_info']['n_dropped']}) | "
          f"G_HIGH n_cols={r['high_summary']['n_cols']} "
          f"mean_abs_phi={r['high_summary']['mean_abs_phi']:.4f} | "
          f"G_LOW n_cols={r['low_summary']['n_cols']} "
          f"mean_abs_phi={r['low_summary']['mean_abs_phi']:.4f}")

# Saglama noktasi: KANSER'de AL_16..25 en yuksek phi'yi vermeli
_kanser_flags = panel_results['KANSER']['flag_df']
_kanser_sorted = _kanser_flags.sort_values('abs_phi', ascending=False)
_top10_cols = set(_kanser_sorted.head(10)['col'])
_leakage_in_top10 = _top10_cols & set(AL_MISSINGNESS_LEAKAGE_RISK)
print(f"\nSaglama noktasi -- KANSER top-10 abs_phi sutunlarindan AL_16..25 kesisimi: "
      f"{len(_leakage_in_top10)}/10 -> {sorted(_leakage_in_top10)}")

# Temizlik ozeti (tum paneller)
print(f"\nSabit + ozdes sutun temizligi ozeti:")
for p in PANELS:
    ci = panel_results[p]['clean_info']
    print(f"  {p}: ham={ci['n_raw_cols']}, sabit={ci['n_constant']}, ozdes_cift={ci['n_dup_pairs']}, "
          f"drop={ci['n_dropped']}, temiz={ci['n_clean_cols']}")


MASTER: temiz_sutun=288 (drop=63) | G_HIGH n_cols=140 mean_abs_phi=0.2357 | G_LOW n_cols=148 mean_abs_phi=0.1363


KANSER: temiz_sutun=280 (drop=71) | G_HIGH n_cols=191 mean_abs_phi=0.4164 | G_LOW n_cols=89 mean_abs_phi=0.2531


PAH: temiz_sutun=258 (drop=93) | G_HIGH n_cols=150 mean_abs_phi=0.0474 | G_LOW n_cols=108 mean_abs_phi=0.0844


CFTR [DUSUK-GUVEN: n_benign=21]: temiz_sutun=279 (drop=72) | G_HIGH n_cols=31 mean_abs_phi=0.2863 | G_LOW n_cols=237 mean_abs_phi=0.0752

Saglama noktasi -- KANSER top-10 abs_phi sutunlarindan AL_16..25 kesisimi: 10/10 -> ['AL_16', 'AL_17', 'AL_18', 'AL_19', 'AL_20', 'AL_21', 'AL_22', 'AL_23', 'AL_24', 'AL_25']

Sabit + ozdes sutun temizligi ozeti:
  MASTER: ham=351, sabit=57, ozdes_cift=583, drop=63, temiz=288
  KANSER: ham=351, sabit=69, ozdes_cift=694, drop=71, temiz=280
  PAH: ham=351, sabit=91, ozdes_cift=886, drop=93, temiz=258
  CFTR: ham=351, sabit=70, ozdes_cift=696, drop=72, temiz=279


In [7]:
# Cell 7: CSV Ciktilari Kaydet
group_summary_rows = []

for p in PANELS:
    r = panel_results[p]

    high_path = os.path.join(RESULTS_DIR, f'{p}_HIGH_flag_stats.csv')
    low_path = os.path.join(RESULTS_DIR, f'{p}_LOW_flag_stats.csv')
    r['g_high'].sort_values('abs_phi', ascending=False).to_csv(high_path, index=False)
    r['g_low'].sort_values('abs_phi', ascending=False).to_csv(low_path, index=False)

    for grp_name, summary in [('G_HIGH', r['high_summary']), ('G_LOW', r['low_summary'])]:
        group_summary_rows.append({
            'panel': p,
            'group': grp_name,
            'n_cols': summary['n_cols'],
            'mean_phi': summary['mean_phi'],
            'mean_abs_phi': summary['mean_abs_phi'],
            'fluctuation': summary['fluctuation'],
            'min_phi': summary['min_phi'],
            'max_phi': summary['max_phi'],
            'mean_delta': summary['mean_delta'],
            'top3_cols': ','.join(summary['top3_cols']),
            'low_confidence': r['low_confidence'],
        })

group_summaries_df = pd.DataFrame(group_summary_rows)
group_summaries_path = os.path.join(RESULTS_DIR, 'group_summaries.csv')
group_summaries_df.to_csv(group_summaries_path, index=False)

print(f'8 CSV kaydedildi: {RESULTS_DIR}/{{PANEL}}_{{HIGH,LOW}}_flag_stats.csv')
print(f'Ozet: {group_summaries_path}')
group_summaries_df


8 CSV kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v26_missing_flag_correlation/{PANEL}_{HIGH,LOW}_flag_stats.csv
Ozet: /Users/tefe/teknofest_model/teknofest_model/results/v26_missing_flag_correlation/group_summaries.csv


,panel,group,n_cols,mean_phi,mean_abs_phi,fluctuation,min_phi,max_phi,mean_delta,top3_cols,low_confidence
0,MASTER,G_HIGH,140,0.235710,0.235710,0.045883,0.093547,0.345526,0.236680,"AL_1,AL_5,AL_6",False
1,MASTER,G_LOW,148,0.118833,0.136252,0.085384,-0.111101,0.208978,0.136948,"AL_300,AL_305,AL_315",False
2,KANSER,G_HIGH,191,0.416399,0.416399,0.060783,0.339605,0.567702,0.409920,"AL_20,AL_22,AL_25",False
3,KANSER,G_LOW,89,0.194304,0.253081,0.223416,-0.245837,0.476840,0.225232,"AL_7,AL_12,AL_8",False
4,PAH,G_HIGH,150,-0.041367,0.047432,0.030728,-0.074745,0.025268,-0.056194,"AL_184,AL_183,AL_41",False
5,PAH,G_LOW,108,0.051486,0.084398,0.109123,-0.164308,0.213374,0.077658,"AL_325,AL_314,AL_317",False
6,CFTR,G_HIGH,31,0.286349,0.286349,0.153954,0.073361,0.414614,0.268356,"AL_1,AL_29,AL_27",True
7,CFTR,G_LOW,237,0.002169,0.075203,0.093801,-0.177803,0.244136,-0.000690,"EK_3,AL_315,AL_296",True


In [8]:
# Cell 8: Görselleştirmeler
# --- Fig 1: panel x grup ortalama |phi| + fluctuation error-bar ---
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(PANELS))
width = 0.35

high_means = [panel_results[p]['high_summary']['mean_abs_phi'] for p in PANELS]
high_errs = [panel_results[p]['high_summary']['fluctuation'] if not np.isnan(panel_results[p]['high_summary']['fluctuation']) else 0 for p in PANELS]
low_means = [panel_results[p]['low_summary']['mean_abs_phi'] for p in PANELS]
low_errs = [panel_results[p]['low_summary']['fluctuation'] if not np.isnan(panel_results[p]['low_summary']['fluctuation']) else 0 for p in PANELS]

ax.bar(x_pos - width/2, high_means, width, yerr=high_errs, label='G_HIGH (>%50 null)', color='#c0392b', capsize=5)
ax.bar(x_pos + width/2, low_means, width, yerr=low_errs, label='G_LOW (<=%50 null)', color='#2980b9', capsize=5)
ax.set_xticks(x_pos)
ax.set_xticklabels(PANELS)
ax.set_ylabel('mean |phi|')
ax.set_title('Panel x Grup: Ortalama |phi| (hata cubugu = fluctuation/std)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig1_path = os.path.join(RESULTS_DIR, 'fig1_group_abs_phi.png')
plt.savefig(fig1_path, dpi=120)
plt.close()

# --- Fig 2: phi dagilimi histogram (G_HIGH vs G_LOW) ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, p in zip(axes.flat, PANELS):
    r = panel_results[p]
    if len(r['g_high']) > 0:
        ax.hist(r['g_high']['phi'], bins=20, alpha=0.6, label='G_HIGH', color='#c0392b')
    if len(r['g_low']) > 0:
        ax.hist(r['g_low']['phi'], bins=20, alpha=0.6, label='G_LOW', color='#2980b9')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(p)
    ax.set_xlabel('phi')
    ax.legend()
fig.suptitle('phi Dagilimi: G_HIGH vs G_LOW')
plt.tight_layout()
fig2_path = os.path.join(RESULTS_DIR, 'fig2_phi_hist.png')
plt.savefig(fig2_path, dpi=120)
plt.close()

# --- Fig 3: KANSER AL_16..25 vurgulu ---
fig, ax = plt.subplots(figsize=(12, 6))
kanser_flags = panel_results['KANSER']['flag_df'].sort_values('abs_phi', ascending=False)
colors = ['#e74c3c' if c in AL_MISSINGNESS_LEAKAGE_RISK else '#95a5a6' for c in kanser_flags['col']]
ax.bar(range(len(kanser_flags)), kanser_flags['phi'], color=colors)
ax.set_xticks(range(len(kanser_flags)))
ax.set_xticklabels(kanser_flags['col'], rotation=90, fontsize=6)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('phi')
ax.set_title('KANSER: Tum Flag phi (kirmizi = AL_16..25 leakage-riski bloğu)')
plt.tight_layout()
fig3_path = os.path.join(RESULTS_DIR, 'fig3_kanser_leakage.png')
plt.savefig(fig3_path, dpi=120)
plt.close()

# --- Fig 4: en guclu 15 sutun (delta siralı, tum paneller) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, p in zip(axes.flat, PANELS):
    flags = panel_results[p]['flag_df'].copy()
    flags['abs_delta'] = flags['delta'].abs()
    top15 = flags.sort_values('abs_delta', ascending=False).head(15)
    colors = ['#c0392b' if d > 0 else '#2980b9' for d in top15['delta']]
    ax.barh(top15['col'], top15['delta'], color=colors)
    ax.set_title(f'{p}: En Guclu 15 Sutun (delta = patho-benign eksiklik farki)')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
plt.tight_layout()
fig4_path = os.path.join(RESULTS_DIR, 'fig4_top_delta.png')
plt.savefig(fig4_path, dpi=120)
plt.close()

print('4 gorsel kaydedildi:')
print(f'  {fig1_path}')
print(f'  {fig2_path}')
print(f'  {fig3_path}')
print(f'  {fig4_path}')


4 gorsel kaydedildi:
  /Users/tefe/teknofest_model/teknofest_model/results/v26_missing_flag_correlation/fig1_group_abs_phi.png
  /Users/tefe/teknofest_model/teknofest_model/results/v26_missing_flag_correlation/fig2_phi_hist.png
  /Users/tefe/teknofest_model/teknofest_model/results/v26_missing_flag_correlation/fig3_kanser_leakage.png
  /Users/tefe/teknofest_model/teknofest_model/results/v26_missing_flag_correlation/fig4_top_delta.png


In [9]:
# Cell 9: Kapsamli PDF Rapor

class NB43Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB43: Missing-Flag Korelasyon Analizi - Tum Paneller', 0, 1, 'C')
        self.set_font('Helvetica', '', 9)
        self.cell(0, 5, f'SEED={SEED} | 4 panel x 2 grup (G_HIGH >%50 null, G_LOW <=%50 null) | sabit+ozdes sutun temizligi uygulandi', 0, 1, 'C')
        self.ln(3)

    def section_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 8, f'  {title}', 0, 1, 'L', fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def body_text(self, text):
        self.set_font('Helvetica', '', 9)
        self.multi_cell(0, 5, text)
        self.ln(2)

    def warning_text(self, text):
        self.set_font('Helvetica', 'B', 9)
        self.set_fill_color(231, 76, 60)
        self.set_text_color(255, 255, 255)
        self.multi_cell(0, 6, text, fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [self.epw / len(headers)] * len(headers)
        self.set_font('Helvetica', 'B', 7)
        self.set_fill_color(52, 73, 94)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), 1, 0, 'C', fill=True)
        self.ln()
        self.set_font('Helvetica', '', 6.5)
        self.set_text_color(0, 0, 0)
        for j, row in enumerate(rows):
            if j % 2 == 0:
                self.set_fill_color(236, 240, 241)
            else:
                self.set_fill_color(255, 255, 255)
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, 'C', fill=True)
            self.ln()
        self.ln(3)


def flag_table_rows(flag_df, max_rows=30):
    '''Bir grup tablosunu PDF satirlarina cevirir (abs_phi sirali, ilk max_rows).'''
    sorted_df = flag_df.sort_values('abs_phi', ascending=False).head(max_rows)
    rows = []
    for _, r in sorted_df.iterrows():
        rows.append([
            r['col'],
            f"{r['miss_ratio']:.3f}",
            f"{r['miss_rate_patho']:.3f}",
            f"{r['miss_rate_benign']:.3f}",
            f"{r['delta']:+.3f}",
            f"{r['phi']:+.3f}",
        ])
    return rows


pdf = NB43Report()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# --- 0. Motivasyon ve Kavram ---
pdf.section_title('0. Motivasyon ve Yontem')
pdf.body_text(
    "Mevcut missing stratejisi M3 (>%50 NaN icin is_missing flag + medyan impute) NB14 panel-transfer\n"
    "ablasyonuyla secildi, ama eksiklik-etiket korelasyonu sistematik olcculmedi.\n\n"
    "Bu notebook: her panelde -- sabit ve ozdes sutunlar drop edildikten sonra -- eksiklik>0 olan her\n"
    "sutun icin is_missing flag uretir, flag ile Label arasindaki phi (phi) katsayisini ve sinif-bazli\n"
    "eksiklik oranlarini hesaplar. Sutunlar miss_ratio>0.50 (G_HIGH) ve miss_ratio<=0.50 (G_LOW) olarak\n"
    "iki gruba ayrilir; her grup icin mean_phi (isaretli), mean_abs_phi (toplam sinyal) ve\n"
    "fluctuation=std(phi) (grup-ici tutarlilik) raporlanir.\n\n"
    "Bu bir olcum, tahmin degildir: train/test split yok, impute yok. Flag'ler temiz+HAM eksiklikten\n"
    "uretildi (impute ONCESI)."
)

# --- 0b. Sabit + Ozdes Sutun Temizligi (bu revizyonda eklendi) ---
pdf.section_title('0b. On Temizlik: Sabit ve Ozdes Sutunlar')
pdf.body_text(
    "CLAUDE.md Sozlesmeler madde 3-4 geregi diger notebooklarda (NB15, NB36-39) uygulanan temizlik burada\n"
    "da uygulandi: sabit sutunlar (nunique<=1) ve ozdes sutun ciftlerinden biri (583 cift, cogu AL\n"
    "bloklarinda) drop edildi. Nedeni: ozdes sutunlar (orn. MASTER'da AL_1..AL_6) ayni missing-flag\n"
    "sinyalinin birebir kopyasidir -- temizlik olmadan 'en yuksek phi' listeleri tek bir sinyalin N kez\n"
    "tekrari gibi gorunur, bagimsiz kanit sayisi abartilmis olur."
)
clean_headers = ['Panel', 'Ham Sutun', 'Sabit', 'Ozdes Cift', 'Drop Toplam', 'Temiz Sutun']
clean_rows = []
for p in PANELS:
    ci = panel_results[p]['clean_info']
    clean_rows.append([
        p, ci['n_raw_cols'], ci['n_constant'], ci['n_dup_pairs'], ci['n_dropped'], ci['n_clean_cols'],
    ])
pdf.add_table(clean_headers, clean_rows, [30, 30, 25, 30, 30, 30])

pdf.section_title('Karar Kurali')
decision_headers = ['Bulgu', 'Anlam', 'Aksiyon']
decision_rows = [
    ['G_LOW mean|phi| yuksek (>~0.15)', "M3 degerli flag'leri kaciriyor", 'Esigi dusur / secici flag seti'],
    ['G_HIGH yuksek + dusuk fluctuation', 'Eksiklik guclu, tutarli sinyal', 'M3 dogru; blogu modelde tut'],
    ['Bir grupta yuksek fluctuation', 'Sinyal birkac sutunda', 'O sutunlari izole et, kalani drop'],
    ['AL_16..25 (KANSER) yuksek phi', 'Bilinen leakage riski teyidi', 'Iki model (ile/olmadan) karsilastir'],
]
pdf.add_table(decision_headers, decision_rows, [70, 65, 55])

# --- 1. Ozet Tablo (8 grup) ---
pdf.add_page()
pdf.section_title('1. Ozet: 4 Panel x 2 Grup')
summary_headers = ['Panel', 'Grup', 'n_cols', 'mean_phi', 'mean_abs_phi', 'fluctuation', 'mean_delta']
summary_rows = []
for _, row in group_summaries_df.iterrows():
    summary_rows.append([
        row['panel'], row['group'], int(row['n_cols']),
        f"{row['mean_phi']:+.4f}" if pd.notna(row['mean_phi']) else 'NaN',
        f"{row['mean_abs_phi']:.4f}" if pd.notna(row['mean_abs_phi']) else 'NaN',
        f"{row['fluctuation']:.4f}" if pd.notna(row['fluctuation']) else 'NaN',
        f"{row['mean_delta']:+.4f}" if pd.notna(row['mean_delta']) else 'NaN',
    ])
pdf.add_table(summary_headers, summary_rows, [25, 20, 20, 30, 30, 30, 30])

pdf.body_text(
    "Saglama noktasi: KANSER'de AL_16..25 (AL_MISSINGNESS_LEAKAGE_RISK) blogu, tum sutunlar arasinda\n"
    f"en yuksek abs_phi degerlerini vermeli. Top-10 kesisim: {len(_leakage_in_top10)}/10 sutun -> "
    f"{sorted(_leakage_in_top10)}"
)

# --- 2. Panel Bazli Detay Tablolar (8 tablo) ---
detail_headers = ['col', 'miss%', 'miss_patho%', 'miss_benign%', 'delta', 'phi']
col_widths = [35, 22, 28, 28, 22, 22]

for p in PANELS:
    r = panel_results[p]
    pdf.add_page()
    title_suffix = ' [DUSUK-GUVEN: n_benign=21]' if r['low_confidence'] else ''
    pdf.section_title(f'2. {p} - G_HIGH (miss_ratio > %50){title_suffix}')

    if r['low_confidence']:
        pdf.warning_text(
            "UYARI: CFTR toplam benign sayisi 21. Sinif-bazli eksiklik oranlari (miss_rate_benign, delta)\n"
            "cok gurultulu -- tek basina guvenilmemeli. Bkz. CLAUDE.md CFTR DEGERLENDIRME GUVENI YOK bolumu."
        )

    high_rows = flag_table_rows(r['g_high'], max_rows=25)
    if high_rows:
        pdf.add_table(detail_headers, high_rows, col_widths)
    else:
        pdf.body_text('Bu grupta sutun yok.')

    hs = r['high_summary']
    pdf.body_text(
        f"Ozet -- n_cols={hs['n_cols']} | mean_phi={hs['mean_phi']:+.4f} | "
        f"mean_abs_phi={hs['mean_abs_phi']:.4f} | fluctuation={hs['fluctuation']:.4f} | "
        f"top3={', '.join(hs['top3_cols'])}"
    )

    pdf.section_title(f'2. {p} - G_LOW (0 < miss_ratio <= %50){title_suffix}')
    low_rows = flag_table_rows(r['g_low'], max_rows=25)
    if low_rows:
        pdf.add_table(detail_headers, low_rows, col_widths)
    else:
        pdf.body_text('Bu grupta sutun yok.')

    ls = r['low_summary']
    pdf.body_text(
        f"Ozet -- n_cols={ls['n_cols']} | mean_phi={ls['mean_phi']:+.4f} | "
        f"mean_abs_phi={ls['mean_abs_phi']:.4f} | fluctuation={ls['fluctuation']:.4f} | "
        f"top3={', '.join(ls['top3_cols'])}"
    )

# --- 3. Gorseller ---
pdf.add_page()
pdf.section_title('3. Gorseller')
for img_name, img_title in [
    ('fig1_group_abs_phi.png', 'Panel x Grup Ortalama |phi| (fluctuation error-bar)'),
    ('fig2_phi_hist.png', 'phi Dagilimi: G_HIGH vs G_LOW'),
    ('fig3_kanser_leakage.png', 'KANSER AL_16..25 Leakage Blogu Vurgulu'),
    ('fig4_top_delta.png', 'Panel Basina En Guclu 15 Sutun (delta sirali)'),
]:
    img_path = os.path.join(RESULTS_DIR, img_name)
    if os.path.exists(img_path):
        pdf.section_title(img_title)
        pdf.image(img_path, x=10, w=190)
        pdf.add_page()

# --- 4. Sonuc ve Karar ---
pdf.section_title('4. Sonuc ve M3 Degerlendirmesi')

# Otomatik karar metni: G_LOW sinyali guclu mu?
low_signal_panels = []
for p in PANELS:
    ls = panel_results[p]['low_summary']
    if pd.notna(ls['mean_abs_phi']) and ls['mean_abs_phi'] > 0.15:
        low_signal_panels.append((p, ls['mean_abs_phi']))

if low_signal_panels:
    panel_list = ', '.join([f'{p} (mean_abs_phi={v:.3f})' for p, v in low_signal_panels])
    pdf.body_text(
        f"BULGU: Su panellerde G_LOW (<=%50 null) grubu da guclu sinyal tasiyor: {panel_list}.\n"
        "SONUC: M3'un %50 esigi bu panellerde degerli flag'leri kaciriyor olabilir. "
        "Esigi dusurmek veya phi-tabanli secici flag seti kurmak icin yeni bir ablasyon (NB44) onerilir."
    )
else:
    pdf.body_text(
        "BULGU: Hicbir panelde G_LOW grubu guclu sinyal gostermedi (tum mean_abs_phi <= 0.15).\n"
        "SONUC: M3'un %50 esigi veriyle DOGRULANDI -- flag uretilen G_HIGH bloklari zaten ana sinyal "
        "tasiyan sutunlar. Esik degisikligine gerek yok."
    )

pdf.body_text(
    "Not: KANSER AL_16..25 blogu (AL_MISSINGNESS_LEAKAGE_RISK) icin bu analiz, blogun eksiklik-etiket "
    "korelasyonunu nicel olarak teyit/reddeder (Bolum 1'deki saglama noktasina bakiniz). CFTR sonuclari "
    "n=21 benign nedeniyle dusuk-guven ile yorumlanmalidir.\n\n"
    "Bu revizyonda sabit+ozdes sutun temizligi uygulandi (Bolum 0b) -- ozdes AL bloklarinin tekrar eden\n"
    "phi degerleri artik grup ortalamalarini yapay olarak sismemektedir."
)

report_path = os.path.join(REPORTS_DIR, 'NB43_missing_flag_correlation_report.pdf')
pdf.output(report_path)
print(f'PDF rapor kaydedildi: {report_path}')


PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB43_missing_flag_correlation_report.pdf


In [10]:
# Cell 10: Final Özet
print('=' * 70)
print('NB43 TAMAMLANDI -- MISSING-FLAG KORELASYON ANALIZI')
print('=' * 70)

print('\nOn temizlik (sabit + ozdes sutun drop):')
for p in PANELS:
    ci = panel_results[p]['clean_info']
    print(f"  {p}: ham={ci['n_raw_cols']} -> temiz={ci['n_clean_cols']} "
          f"(sabit={ci['n_constant']}, ozdes_cift={ci['n_dup_pairs']}, toplam_drop={ci['n_dropped']})")

for p in PANELS:
    r = panel_results[p]
    lc = ' [DUSUK-GUVEN]' if r['low_confidence'] else ''
    print(f"\n{p}{lc} (n={r['n_rows']}, pos={r['n_pos']}, neg={r['n_neg']}):")
    print(f"  G_HIGH: n_cols={r['high_summary']['n_cols']}, mean_abs_phi={r['high_summary']['mean_abs_phi']:.4f}, "
          f"fluctuation={r['high_summary']['fluctuation']:.4f}")
    print(f"  G_LOW : n_cols={r['low_summary']['n_cols']}, mean_abs_phi={r['low_summary']['mean_abs_phi']:.4f}, "
          f"fluctuation={r['low_summary']['fluctuation']:.4f}")

print(f"\nSaglama noktasi -- KANSER top-10 abs_phi / AL_16..25 kesisimi: {len(_leakage_in_top10)}/10")

print(f'\nCiktilar:')
print(f'  8 CSV: {RESULTS_DIR}/{{PANEL}}_{{HIGH,LOW}}_flag_stats.csv')
print(f'  Ozet CSV: {RESULTS_DIR}/group_summaries.csv')
print(f'  4 PNG: {RESULTS_DIR}/fig{{1,2,3,4}}_*.png')
print(f'  PDF: {REPORTS_DIR}/NB43_missing_flag_correlation_report.pdf')


NB43 TAMAMLANDI -- MISSING-FLAG KORELASYON ANALIZI

On temizlik (sabit + ozdes sutun drop):
  MASTER: ham=351 -> temiz=288 (sabit=57, ozdes_cift=583, toplam_drop=63)
  KANSER: ham=351 -> temiz=280 (sabit=69, ozdes_cift=694, toplam_drop=71)
  PAH: ham=351 -> temiz=258 (sabit=91, ozdes_cift=886, toplam_drop=93)
  CFTR: ham=351 -> temiz=279 (sabit=70, ozdes_cift=696, toplam_drop=72)

MASTER (n=2931, pos=2149, neg=782):
  G_HIGH: n_cols=140, mean_abs_phi=0.2357, fluctuation=0.0459
  G_LOW : n_cols=148, mean_abs_phi=0.1363, fluctuation=0.0854

KANSER (n=388, pos=268, neg=120):
  G_HIGH: n_cols=191, mean_abs_phi=0.4164, fluctuation=0.0608
  G_LOW : n_cols=89, mean_abs_phi=0.2531, fluctuation=0.2234

PAH (n=372, pos=310, neg=62):
  G_HIGH: n_cols=150, mean_abs_phi=0.0474, fluctuation=0.0307
  G_LOW : n_cols=108, mean_abs_phi=0.0844, fluctuation=0.1091

CFTR [DUSUK-GUVEN] (n=111, pos=90, neg=21):
  G_HIGH: n_cols=31, mean_abs_phi=0.2863, fluctuation=0.1540
  G_LOW : n_cols=237, mean_abs_phi=0.